In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, isnull
from pyspark.sql.functions import col, year, month, regexp_extract, split, sum as spark_sum, isnull, expr
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

In [2]:
spark = (
    SparkSession.builder
    .appName("Data Preprocessing Spark Application")
    .master("spark://100.127.25.114:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 21:56:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Read dataset

In [4]:
path = "hdfs://100.127.25.114:9000/data-src/raw_sgp_flat_resale.csv" 

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path)
)

print(f"Dataset shape: ({df.count()}, {len(df.columns)})")
print("Columns:", len(df.columns))
df.printSchema()
df.show(10, truncate=False)

Dataset shape: (181262, 11)
Columns: 11
root
 |-- month: timestamp (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- remaining_lease: string (nullable = true)
 |-- resale_price: double (nullable = true)

+-------------------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------------+------------+
|month              |town      |flat_type|block|street_name      |storey_range|floor_area_sqm|flat_model    |lease_commence_date|remaining_lease   |resale_price|
+-------------------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------------+----------

### Data Quality Assessment

In [5]:
# Check Duplicate
print("\n---  Check Duplicate ---")
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
duplicate_count = total_rows - distinct_rows
print(f"Tổng số dòng: {total_rows}")
print(f"Số dòng sau khi xóa trùng: {distinct_rows}")
print(f"-> Số dòng trùng lặp (Duplicate): {duplicate_count}")


---  Check Duplicate ---


Tổng số dòng: 181262
Số dòng sau khi xóa trùng: 180974
-> Số dòng trùng lặp (Duplicate): 288


In [6]:
# Check Null
print("\n--- Check Null ---")
df.select([spark_sum(isnull(c).cast("int")).alias(c) for c in df.columns]).show()


--- Check Null ---
+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+---------------+------------+
|month|town|flat_type|block|street_name|storey_range|floor_area_sqm|flat_model|lease_commence_date|remaining_lease|resale_price|
+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+---------------+------------+
|    0|   0|        0|    0|          0|           0|             0|         0|                  0|              0|           0|
+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+---------------+------------+



In [7]:
# Check Outlier 
print("\n---  Check Outlier ---")
df.select("resale_price", "floor_area_sqm", "lease_commence_date") \
  .summary("min", "25%", "50%", "75%", "max").show()


---  Check Outlier ---
+-------+------------+--------------+-------------------+
|summary|resale_price|floor_area_sqm|lease_commence_date|
+-------+------------+--------------+-------------------+
|    min|    140000.0|          31.0|               1966|
|    25%|    370000.0|          82.0|               1985|
|    50%|    468000.0|          93.0|               1996|
|    75%|    592000.0|         112.0|               2010|
|    max|   1588000.0|         249.0|               2020|
+-------+------------+--------------+-------------------+



### Data Preprocessing

In [8]:
# Remove Duplicate
df_dedup = df.dropDuplicates()
print("Số dòng sau khi loại trùng lặp:", df_dedup.count())

df_eng = df_dedup.withColumn("transaction_year", year(col("month"))) \
                 .withColumn("transaction_month", month(col("month")))

df_eng = df_eng.withColumn("lease_years_raw", regexp_extract(col("remaining_lease"), r"(\d+)\s*years?", 1)) \
                 .withColumn("lease_years", expr("try_cast(lease_years_raw as int)"))

df_eng = df_eng.withColumn("lease_months_raw", regexp_extract(col("remaining_lease"), r"(\d+)\s*months?", 1)) \
                 .withColumn("lease_months_part", expr("try_cast(lease_months_raw as int)"))

# Handle Missing Values
df_eng = df_eng.fillna({"lease_years": 0, "lease_months_part": 0})

Số dòng sau khi loại trùng lặp: 180974


### Feature Engineering

In [9]:
# Remaining Lease (Months)
df_eng = df_eng.withColumn("remaining_lease_months", (col("lease_years") * 12) + col("lease_months_part"))

# Flat Age
df_eng = df_eng.withColumn("flat_age", col("transaction_year") - col("lease_commence_date"))

# Average Storey Level
df_eng = df_eng.withColumn("storey_lower_raw", split(col("storey_range"), " TO ")[0]) \
                 .withColumn("storey_upper_raw", split(col("storey_range"), " TO ")[1]) \
                 .withColumn("storey_lower", expr("try_cast(storey_lower_raw as int)")) \
                 .withColumn("storey_upper", expr("try_cast(storey_upper_raw as int)")) \
                 .withColumn("storey_avg", (col("storey_lower") + col("storey_upper")) / 2.0)

df_clean = df_eng.drop(
    "month", "remaining_lease", "lease_years_raw", "lease_years", 
    "lease_months_raw", "lease_months_part", "storey_range", 
    "storey_lower_raw", "storey_upper_raw", "storey_lower", "storey_upper",
    "block", "street_name"
)

df_clean.cache()
df_clean.count()


180974

In [10]:
print("\n=== 1. Kiểm tra Null sau khi xử lý đặc trưng ===")
df_clean.select([spark_sum(isnull(c).cast("int")).alias(c) for c in df_clean.columns]).show()

print("=== 2. Kiểm tra phân phối / Range check feature mới ===")
df_clean.select("remaining_lease_months", "flat_age", "storey_avg") \
        .summary("min", "25%", "50%", "75%", "max").show()

print("=== 3. Cấu trúc Schema cuối cùng ===")
df_clean.printSchema()

print("\n=== 4. Xem trước 5 dòng dữ liệu mẫu ===")
df_clean.show(5, truncate=False)
print(f"Kích thước tập dữ liệu hoàn thiện: ({df_clean.count()}, {len(df_clean.columns)})")


=== 1. Kiểm tra Null sau khi xử lý đặc trưng ===


+----+---------+--------------+----------+-------------------+------------+----------------+-----------------+----------------------+--------+----------+
|town|flat_type|floor_area_sqm|flat_model|lease_commence_date|resale_price|transaction_year|transaction_month|remaining_lease_months|flat_age|storey_avg|
+----+---------+--------------+----------+-------------------+------------+----------------+-----------------+----------------------+--------+----------+
|   0|        0|             0|         0|                  0|           0|               0|                0|                     0|       0|         0|
+----+---------+--------------+----------+-------------------+------------+----------------+-----------------+----------------------+--------+----------+

=== 2. Kiểm tra phân phối / Range check feature mới ===


+-------+----------------------+--------+----------+
|summary|remaining_lease_months|flat_age|storey_avg|
+-------+----------------------+--------+----------+
|    min|                   498|       1|       2.0|
|    25%|                   758|      11|       5.0|
|    50%|                   894|      25|       8.0|
|    75%|                  1060|      36|      11.0|
|    max|                  1173|      58|      50.0|
+-------+----------------------+--------+----------+

=== 3. Cấu trúc Schema cuối cùng ===
root
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- resale_price: double (nullable = true)
 |-- transaction_year: integer (nullable = true)
 |-- transaction_month: integer (nullable = true)
 |-- remaining_lease_months: integer (nullable = false)
 |-- flat_age: integer (nullable = true)
 |-- storey_avg: double

### Train/Test Split

In [11]:
# Chia train,test
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Số dòng tập Train: {train_df.count()}")
print(f"Số dòng tập Test:  {test_df.count()}")

Số dòng tập Train: 144887
Số dòng tập Test:  36087


### Spark ML Pipeline

In [12]:
categorical_cols = ["town", "flat_type", "flat_model"]
numeric_cols = ["floor_area_sqm", "transaction_year", 
                "transaction_month", "remaining_lease_months", "flat_age", "storey_avg"]

# StringIndexer
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical_cols]

# OneHotEncoder
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_vec") for c in categorical_cols]

# VectorAssembler
assembler_inputs = [c+"_vec" for c in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features_raw")

# StandardScaler
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)


pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
print("\nĐang xử lý Pipeline... ")
pipeline_model = pipeline.fit(train_df)

train_transformed = pipeline_model.transform(train_df)
test_transformed = pipeline_model.transform(test_df)

print("\n=== Vector Features sau khi xử lý qua Pipeline ===")
train_transformed.select("features", "resale_price").show(5, truncate=False)

print("Train Schema")
train_transformed.printSchema()


Đang xử lý Pipeline... 



=== Vector Features sau khi xử lý qua Pipeline ===
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [13]:
train_df.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("hdfs://100.127.25.114:9000/data-src/train_csv")

In [14]:
test_df.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("hdfs://100.127.25.114:9000/data-src/test_csv")

In [15]:
spark.stop()